# [전체 설명]
- 33류 해설서 json 파일(ex_33.json) 청크 구분 안하기
- collection 이름은 HS_33_JSON_4
- 품목분류서 청크 사이즈 2000 -> 1000으로 줄이기

In [1]:
import json

with open('./ex_33.json', 'r', encoding='utf-8') as f:
    ex = json.load(f)

# 33류 테스트할 새로운 collection(HS_33_JSON_3) 만들기

In [2]:
# API 키 미리 지정하기
MY_API = ""

## 패키지 로드

In [3]:
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
import chromadb

## client 호출

In [5]:
client = chromadb.PersistentClient(path='../Chroma DB/chroma_db_fixed')
client.heartbeat()

1745237880036814700

In [6]:
from chromadb.utils import embedding_functions
from chromadb.utils.embedding_functions import OpenAIEmbeddingFunction

embedding_function = OpenAIEmbeddingFunction(api_key = MY_API, model_name='text-embedding-3-small')

collection = client.create_collection(name="HS_33_JSON_4", embedding_function=embedding_function)

In [7]:
# collection 만들어졌나 확인하기
collections = client.list_collections()
for name in collections:
    print(name)

HS_33_2000
HS_33_JSON
HS_33_JSON_3
Trit
HS_33_JSON_2000
Practice
HS_33
HS_33_JSON_4


In [8]:
ids = [f"33_ex_{i}" for i in range(len(ex))]

In [10]:
from langchain.schema import Document

documents = [
    Document(page_content=item["content"], metadata=item.get("metadata", {}))
    for item in ex
]

# 랭체인 크로마를 쓰기 위해서는 반드시 필요
embeddings = OpenAIEmbeddings(api_key=MY_API, model='text-embedding-3-small')

# 'HS_33_JSON' collection에 저장하기
db = Chroma.from_documents(
    documents=documents,
    ids = ids,
    embedding=embeddings,
    persist_directory='../Chroma DB/chroma_db_fixed',
    collection_name='HS_33_JSON_4'
)

In [11]:
len(db.get()['ids'])

136

# collection 추가 자료_품목분류 사례
- 일단 임의로 품목 분류 사례에서는 count 안넣음

In [12]:
import pickle
with open('../HS 코드 분류/품목분류 테스트/samples_list.pkl', 'rb') as f:
    case = pickle.load(f)

In [13]:
case = '\n\n'.join(case)

In [14]:
# 청크로 나누기
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,       # 단어별로 문맥 이어지는게 아니므로 오버랩 0
    length_function=len
)

# 품목분류사례 분할하기
chunks = text_splitter.split_text(case)

In [15]:
len(chunks)

572

In [16]:
# 메타데이터의 갯수는 청크(content)의 갯수와 동일해야 함
metadatas = [{"source": "품목 분류 사례", "count":"정보 없음"}] * len(chunks)
ids = [f"sample_{i}" for i in range(len(chunks))]

In [17]:
# 'HS_33_2000' collection에 저장하기
db = Chroma.from_texts(
    texts = chunks,
    ids = ids,
    embedding=embeddings,
    metadatas=metadatas,
    persist_directory='../Chroma DB/chroma_db_fixed',
    collection_name='HS_33_JSON_4'
)

In [18]:
len(db.get()['ids'])

708

# LLM 요청

## retriever 지정: 검색기로 내가 만든 db 사용

In [19]:
retriever = db.as_retriever()

## 프롬프트_히스토리 저장하기

In [20]:
# prompt 설정
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain.chains import create_history_aware_retriever, create_retrieval_chain
from langchain.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_openai import ChatOpenAI        

In [21]:
# 채팅 히스토리와 질문 통합 함수

# 시스템 프롬프트(지시사항)
contextualize_q_system_prompt = """Given a chat history and the latest user question \
which might reference context in the chat history, formulate a standalone question, \
which can be understood without the chat history. Do NOT answer the question, \
just reformulate it if needed and otherwise return it as is."""

# ChatPromptTemplate 만들기
    # 구성: 시스템 프롬프트 - MessagesPlaceholder(히스토리) - 사용자 프롬프트
contextualize_q_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", contextualize_q_system_prompt),
        MessagesPlaceholder("chat_history"),
        ("human", "{input}"),
    ]
)

# retriever(검색기)에 llm + retrievre + ChatPromptTemplate 넣기

llm = ChatOpenAI(
    temperature=0,
    openai_api_key=MY_API,
    max_tokens=3000,
    model_name="gpt-3.5-turbo",
    request_timeout=120
)

history_aware_retriever = create_history_aware_retriever(llm, retriever, contextualize_q_prompt)

## 체인 만들기

In [22]:
from langchain_openai import ChatOpenAI                                # openai API 사용
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain import hub
from langchain.embeddings import OpenAIEmbeddings

In [23]:
from langchain.prompts import PromptTemplate

# qa_system_prompt_ver1
# qa_system_prompt = """You are an assistant for question-answering tasks. \
# Answer strictly based on the context below. \
# **Do not use any outside knowledge.** \
# If you don't know the answer, just say that you don't know. \
# If the source is 'HS 해설서', and the user's question is ambiguous or lacks enough detail, ask a follow-up question to clarify. \
# Use ten sentences maximum and keep the answer concise. \

# {context}"""

# qa_system_prompt_ver2
qa_system_prompt = """You are an assistant for HS code classification.
Answer based strictly on the context provided.
**Do not use any outside knowledge.**
When answering, consider that documents with higher count values in their metadata are more likely to be accurate or contain the correct answer.
**Do not use any external knowledge or prior training.**
If the document source is 'HS 해설서' and the context suggests ambiguity (e.g. multiple types of gloves or unclear usage),
then politely ask the user for additional information **after** providing your best possible answer.
Keep answers concise and under 10 sentences.

{context}
"""

qa_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", qa_system_prompt),
        MessagesPlaceholder("chat_history"),
        ("human", "{input}")
    ]
)

document_prompt = PromptTemplate.from_template(
    "[출처: {source}]\n{count}\n{page_content}"
)

question_answer_chain = create_stuff_documents_chain(
    llm=llm,
    prompt=qa_prompt,
    document_prompt=document_prompt)


# count 메타데이터가 없는 레코드 존재하므로 넣어주어야 하는 함수
# def format_doc(doc):
#     count = doc.metadata.get("count", "N/A")
#     source = doc.metadata.get("source", "Unknown")
#     return f"[출처: {source}] (Count: {count})\n{doc.page_content}"

# question_answer_chain = create_stuff_documents_chain(
#     llm=llm,
#     prompt=qa_prompt,
#     document_prompt=format_doc  # 유연하게 메타데이터 처리
# )


# 히스토리+질문 검색 retriever와 전달받은 프롬프트를 묶어 llm에 전달하는 체인 하나로 묶기(rag_chain)
rag_chain = create_retrieval_chain(history_aware_retriever, question_answer_chain)

## 채팅 세션별 기록 자동 저장 RAG 체인 구축
- 세션별 기록 자동 저장 아님

In [24]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

### 테스트 1

In [25]:
from langchain_core.messages import HumanMessage

# 채팅 히스토리 적재 위한 리스트
chat_history = []

for i in range(5):
    # chat_history = chat_history[-5:]   # 이전 5개 질문만 기억하기
    question = input("질문: ")
    ai_msg = rag_chain.invoke({"input":question, "chat_history":chat_history})
    # 질문과 답변 히스토리로 저장
    chat_history.extend([HumanMessage(content=question), ai_msg["answer"]])
    print("답변: ", ai_msg["answer"])
    print()

질문:  색조 화장품의 hs 코드가 궁금해


답변:  주어진 정보에는 색조 화장품에 대한 HS 코드에 대한 명시적인 언급이 없습니다. 색조 화장품은 주로 화장품 및 비누 산업에 속하며, 주요 성분은 알코올과 향료입니다. 따라서 색조 화장품은 HS 코드 3307490000에 분류될 수 있습니다. 그러나 보다 정확한 분류를 위해서는 더 구체적인 정보가 필요할 수 있습니다. 부가 정보가 있을 경우 해당 정보를 제공해 주시면 더 정확한 답변을 드릴 수 있습니다.



질문:  립스틱


답변:  주어진 정보에는 "립스틱"에 대한 명시적인 언급이 없습니다. 그러나 립스틱은 색조 화장품에 속하며, 주로 알코올과 향료가 사용됩니다. 따라서 립스틱은 HS 코드 3307490000에 해당하는 색조 화장품으로 분류될 수 있습니다. 더 구체적인 정보가 있을 경우 해당 정보를 제공하시면 보다 정확한 답변을 제공할 수 있습니다.



질문:  그럼 아이섀도우는?


답변:  주어진 정보에는 "아이섀도우"에 대한 명시적인 언급이 없습니다. 아이섀도우는 눈 주변에 사용되는 화장품으로, 색조 화장품에 속합니다. 주로 알코올과 향료가 사용되며, 일부 제품에는 보조적인 의약 성분도 포함될 수 있습니다. 따라서 아이섀도우는 HS 코드 3307490000에 해당하는 색조 화장품으로 분류될 수 있습니다. 더 구체적인 정보가 있을 경우 해당 정보를 제공하시면 보다 정확한 답변을 제공할 수 있습니다.



질문:  혹시 썬크림의 hs 코드도 알려줄 수 있을까?


답변:  주어진 정보에는 "썬크림"에 대한 명시적인 언급이 없습니다. 썬크림은 일반적으로 화장품 및 피부 보호용품으로 분류됩니다. 주로 썬크림은 피부를 자외선으로부터 보호하기 위해 사용되며, 주 성분은 화학 차단제나 물리적 차단제일 수 있습니다. 썬크림은 HS 코드 3304999000 또는 3304991000에 해당하는 화장품으로 분류될 수 있습니다. 더 구체적인 정보가 있을 경우 해당 정보를 제공하시면 보다 정확한 답변을 제공할 수 있습니다.



질문:  립스틱과 아이섀도우가 몇 호로 분류되는지 알려줄래?


답변:  립스틱과 아이섀도우는 모두 색조 화장품에 속하며, 주로 알코올과 향료가 사용됩니다. 따라서 립스틱과 아이섀도우는 모두 HS 코드 3307490000에 해당하는 색조 화장품으로 분류될 수 있습니다.



In [26]:
retriever.invoke('립스틱')

[Document(id='sample_297', metadata={'count': '정보 없음', 'source': '품목 분류 사례'}, page_content='3304999000호 품목 분류 사례 품명: ""Petitfee gold neck cream ; R. KOREA"\n상품 해설: 이 제품은 스쿠알란, 시어버터, 세신추출물, 아데노신, 금박분, 글리세린, 디메치콘, 세테아릴알코올, 물 등으로 조제한 미백색 크림상 50g을 수지제 튜브에 담아 종이제 박스에 소매포장한 것입니다. 이 제품의 용도는 목주름을 개선하기 위한 크림으로 사용됩니다.\n품목 분류 결정 사유: 미용이나 메이크업용 제품류와 기초화장용 제품류에는 선스크린과 선탠 제품류도 포함되며, 매니큐어용 제품류와 페디큐어용 제품류가 관세율표 제3304호에 분류됩니다. 이에 해당하는 미생백색 크림상 50g은 스쿠알란, 시어버터, 세신추출물, 아데노신, 금박분, 글리세린, 디메치콘, 세테아릴알코올, 물 등으로 조제되어 있으며, 주름개선을 위한 기능성 화장품으로 판단하여, 제3304999000호에 분류됩니다. 해당 조항은 관세율표의 해석에 관한 통칙 제1호 및 제6호에 따라 결정되었습니다.'),
 Document(id='sample_401', metadata={'count': '정보 없음', 'source': '품목 분류 사례'}, page_content='3307301000호 품목 분류 사례 품명: ""Perfumed bath salt (Marine Gold)"\n상품 해설: 염화나트륨이 90.5%로, 중탄산나트륨이 7.7%를 차지하며, 주성분에 해조엑기스, 향료, 색소 등이 함유된 황색을 띤 분말을 비닐 봉지에 소매포장한 제품입니다. (총 중량 50g) 해당 제품은 주로 조리 및 조리재료로 사용됩니다.\n품목 분류 결정 사유: HSK 3307301000에 특게된 가향한 목욕염은 신경계계 염류에 속하고, 활동성 물질함량 30% 이상의 이마사 대에 의해 혼성되어 있는 것일 경우 당분간 HSK 33073

### 테스트 2

In [27]:
from langchain_core.messages import HumanMessage

# 채팅 히스토리 적재 위한 리스트
chat_history = []

for i in range(5):
    # chat_history = chat_history[-5:]   # 이전 5개 질문만 기억하기
    question = input("질문: ")
    ai_msg = rag_chain.invoke({"input":question, "chat_history":chat_history})
    # 질문과 답변 히스토리로 저장
    chat_history.extend([HumanMessage(content=question), ai_msg["answer"]])
    print("답변: ", ai_msg["answer"])
    print()

질문:  페퍼민트 오일은 몇 호로 분류돼?


답변:  페퍼민트 오일은 HSK3301240000호로 분류됩니다.



질문:  왜 그렇게 분류되는지 구체적으로 설명해줄래?


답변:  죄송합니다. 추가 정보가 필요합니다. 부연 설명을 위해 다음 문장을 참고해 주십시오: "레몬오일 및 기타 방향성 물질로 구성된 혼합물은 에탄올과 같은 물질로 혼합하여 무색이고 투명한 액상이다. 이 액상은 음료 공업에서 첨가제로 사용되며 일반적으로 사용량은 0.05~0.11%이다."



질문:  3301호는 어떤 규정을 담고 있는지 더 알려줘


답변:  HS 코드 3301은 "정유와 레지노이드, 조제향료, 화장품류, 화장용품류"를 다루는 범주입니다. 이 범주에는 정유(essential oil)와 레지노이드(resinoid), 조제향료, 화장품 및 화장용품이 포함됩니다. 이 범주에서는 천연 올레오레진(oleoresin)이나 특정 식물성 추출물(extract)과 같은 항목들이 제외되며, 특정 화장품 및 화장용품에 대한 세부적인 규정이 포함되어 있습니다. 이 범주는 화장품 및 화장용품에 대한 규정을 제시하고 있으며, 세부적인 화장품 및 화장용품의 분류와 사용 용도에 대한 내용을 다루고 있습니다.



질문:  클렌징 오일은 hs 코드가 뭐야?


답변:  클렌징 오일은 HS 코드 3304.99.90.00에 분류됩니다. 이 코드는 화장품 및 피부 관리용 액체 상의 청정 작용을 가진 제품에 해당합니다.



질문:  네가 말한 답변의 근거는 뭐야? 출처가 확실해?


답변:  죄송합니다. 제가 제공한 답변은 주어진 문맥에서의 일반적인 규정을 기반으로 한 것이며, 특정 출처에 근거한 것은 아닙니다. 더 정확한 정보를 위해서는 해당 제품에 대한 세부적인 내용을 확인해야 합니다.



### 테스트 3 _ gpt 질문

In [28]:
from langchain_core.messages import HumanMessage

# 채팅 히스토리 적재 위한 리스트
chat_history = []

for i in range(10):
    # chat_history = chat_history[-5:]   # 이전 5개 질문만 기억하기
    question = input("질문: ")
    ai_msg = rag_chain.invoke({"input":question, "chat_history":chat_history})
    # 질문과 답변 히스토리로 저장
    chat_history.extend([HumanMessage(content=question), ai_msg["answer"]])
    print("답변: ", ai_msg["answer"])
    print()

질문:  립스틱의 hs 코드는 뭐야?


답변:  주어진 정보에는 립스틱에 대한 HS 코드에 대한 정보가 포함되어 있지 않습니다. 립스틱은 화장품에 속하므로 HS 코드는 3304호에 해당합니다.



질문:  스킨토너나 클렌징 제품은 몇 번 코드로 분류돼?


답변:  주어진 정보에 따르면, 스킨토닉이나 클렌징크림과 같은 제품은 HS 코드 3304에 해당합니다.



질문:  마스카라나 아이섀도우같은 눈 화장품은 어떤 hs 코드로 보내야해?


...

답변:  주어진 정보에 따르면, 매니큐어는 HS 코드 3304301000에, 네일 리무버는 HS 코드 3307909000에 해당합니다.



질문:  고체 방향제같이 고체 상태로 만든 향은 화장품류로 들어가?


답변:  주어진 정보에 따르면, 고체 방향제는 화장품으로 분류됩니다. 해당 제품은 HS 코드 3307909000에 해당할 수 있습니다.



질문:  화장품이랑 의약품을 구분하는 기준이 뭐야? hs 코드 분류 기준이 있어?


답변:  주어진 정보에는 화장품과 의약품을 명확히 구분하는 기준에 대한 언급이 없습니다. 일반적으로, 화장품은 주로 피부 미용이나 개선을 위한 제품으로 사용되며, 의약품은 질병이나 질병 증상의 치료를 목적으로 사용됩니다. HS 코드 분류는 품목의 성질과 용도에 따라 결정되며, 각 품목에 대한 세부 기준은 관세율표 및 해당 규정에 근거하여 정해집니다.



### 테스트 4 _ 모르는 거 물어보기

In [29]:
from langchain_core.messages import HumanMessage

# 채팅 히스토리 적재 위한 리스트
chat_history = []

for i in range(3):
    # chat_history = chat_history[-5:]   # 이전 5개 질문만 기억하기
    question = input("질문: ")
    ai_msg = rag_chain.invoke({"input":question, "chat_history":chat_history})
    # 질문과 답변 히스토리로 저장
    chat_history.extend([HumanMessage(content=question), ai_msg["answer"]])
    print("답변: ", ai_msg["answer"])
    print()

질문:  라벤더 오일은 몇 번으로 분류돼?


답변:  라벤더 오일은 HSK3301230000호로 분류됩니다.



질문:  메이크업 제품의 hs 코드를 알려줄래?


답변:  메이크업 제품은 HS 코드 3304에 해당합니다.



질문:  얼굴에 붙이는 마스크팩인데, 주름 개선 기능이 있어


답변:  얼굴에 붙이는 주름 개선 기능이 있는 마스크 팩은 HS 코드 3304999000에 분류됩니다.



### 테스트 5

In [30]:
from langchain_core.messages import HumanMessage

# 채팅 히스토리 적재 위한 리스트
chat_history = []

for i in range(3):
    # chat_history = chat_history[-5:]   # 이전 5개 질문만 기억하기
    question = input("질문: ")
    ai_msg = rag_chain.invoke({"input":question, "chat_history":chat_history})
    # 질문과 답변 히스토리로 저장
    chat_history.extend([HumanMessage(content=question), ai_msg["answer"]])
    print("답변: ", ai_msg["answer"])
    print()

질문:  인조손톱의 hs 코드가 몇 호인지 알려줄래?


답변:  제공된 정보에는 인조손톱에 대한 HS 코드에 대한 정보가 없습니다. 인조손톱은 특정 HS 코드로 분류되어 있지 않을 수 있습니다. 더 구체적인 정보가 필요하니 추가 정보를 제공해주시면 도움을 드리겠습니다.



질문:  손톱에 장식으로 붙이는건데, 플라스틱으로 만든 가짜 손톱이야


답변:  인조 손톱은 플라스틱으로 만들어진 손톱으로, 일반적으로 화장품 또는 화장용품으로 분류됩니다. 따라서 해당 제품은 HS 코드 3304.99에 해당할 수 있습니다. 하지만 더 정확한 분류를 위해서는 해당 제품의 재질, 용도, 형태 등에 대한 더 자세한 정보가 필요합니다.



질문:  습진 치료나 여드름 치료에 쓰이는 크림도 화장품에 해당되는지 궁금해


답변:  습진 치료나 여드름 치료에 사용되는 크림은 의약품으로 분류될 수 있습니다. 일반적으로 의약품은 화장품이 아닌 별도의 범주로 분류됩니다. 따라서, 해당 제품은 화장품이 아닌 의약품으로 분류될 가능성이 높습니다. 더 정확한 분류를 위해서는 해당 제품의 성분, 용도, 효능 등에 대한 더 자세한 정보가 필요합니다.



### 테스트 6_gpt 2번째 예상질문

In [31]:
from langchain_core.messages import HumanMessage

# 채팅 히스토리 적재 위한 리스트
chat_history = []

for i in range(5):
    # chat_history = chat_history[-5:]   # 이전 5개 질문만 기억하기
    question = input("질문: ")
    ai_msg = rag_chain.invoke({"input":question, "chat_history":chat_history})
    # 질문과 답변 히스토리로 저장
    chat_history.extend([HumanMessage(content=question), ai_msg["answer"]])
    print("답변: ", ai_msg["answer"])
    print()

질문:  립스틱의 HS 코드는 뭐야?


답변:  주어진 정보에는 립스틱에 대한 명시적인 언급이 없습니다. 따라서 립스틱의 HS 코드를 정확히 파악하기 어렵습니다. 립스틱은 화장품으로 분류될 수 있으며, 화장품은 일반적으로 HS 코드 3304에 속합니다. 하지만 더 정확한 분류를 위해서는 립스틱의 성분, 용도, 형태 등에 대한 추가 정보가 필요합니다. 부가 정보를 제공해주시면 보다 정확한 답변을 드릴 수 있습니다.



질문:  샴푸는 몇 류에 속하고, 코드가 어떻게 돼?


답변:  샴푸는 두발용 제품류에 속하며, HS 코드는 330510입니다.



질문:  썬크림은 어떤 코드로 분류돼?


답변:  썬크림은 화장품으로 분류되며, 일반적으로 HS 코드 3304에 속합니다.



질문:  마스크팩은 화장품으로 분류돼?


답변:  네, 마스크팩은 화장품으로 분류됩니다. 일반적으로 마스크팩은 화장품을 부직포에 침투 또는 도포한 제품으로 간주되어 HS 코드 3304에 속합니다.



질문:  화장품 샘플을 수출하려면 HS 코드도 정식 제품이랑 같아?


답변:  화장품 샘플의 HS 코드는 정식 제품과 동일할 수 있습니다. 일반적으로 화장품 샘플은 해당 화장품의 정식 제품과 동일한 성분 및 용도를 가지고 있기 때문에 동일한 HS 코드로 분류될 수 있습니다. 그러나 세부적인 분류는 샘플의 형태, 용도, 성분 등에 따라 달라질 수 있습니다. 따라서 정확한 분류를 위해서는 샘플의 구체적인 특징을 고려해야 합니다.

